## Step 1. Imports

In [0]:
import sys
import os

from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../.."))

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import (
    GOLD_CONTRACTS,
    SILVER_MOVIE,
    SILVER_MOVIE_GENRE,
    SILVER_MOVIE_KEYWORD,
    SILVER_MOVIE_PRODUCTION_COMPANY,
    SILVER_MOVIE_PRODUCTION_COUNTRY,
    SILVER_MOVIE_SPOKEN_LANGUAGE,
    SILVER_CAST_CREDIT,
    SILVER_CREW_CREDIT,
)

## Step 2. Configuração

Resolve os namespaces Silver e Gold a partir da configuração fornecida pelo ambiente

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    silver_schema=dbutils.widgets.get("silver_schema"),
    gold_schema=dbutils.widgets.get("gold_schema"),
)

## Step 3. Leitura Silver

Carrega as entidades Silver necessárias para construir os produtos Gold

In [0]:
movie_df = spark.table(f"{config.silver_namespace}.{SILVER_MOVIE.name}")

movie_genre_df = spark.table(f"{config.silver_namespace}.{SILVER_MOVIE_GENRE.name}")

movie_keyword_df = spark.table(f"{config.silver_namespace}.{SILVER_MOVIE_KEYWORD.name}")

movie_production_company_df = spark.table(
    f"{config.silver_namespace}.{SILVER_MOVIE_PRODUCTION_COMPANY.name}"
)

movie_production_country_df = spark.table(
    f"{config.silver_namespace}.{SILVER_MOVIE_PRODUCTION_COUNTRY.name}"
)

movie_spoken_language_df = spark.table(
    f"{config.silver_namespace}.{SILVER_MOVIE_SPOKEN_LANGUAGE.name}"
)

cast_credit_df = spark.table(f"{config.silver_namespace}.{SILVER_CAST_CREDIT.name}")

crew_credit_df = spark.table(f"{config.silver_namespace}.{SILVER_CREW_CREDIT.name}")

## Step 4. Métricas compartilhadas

Deriva as métricas de filme reutilizadas pelos produtos Gold conforme as regras reconciliadas em `_eda/gold_contract_reconciliation`

In [0]:
movie_metrics_df = movie_df.select(
    "*",
    F.year("release_date").alias("release_year"),
    (F.col("budget") > 0).alias("commercial_metrics_eligible"),
    F.when(
        F.col("budget") > 0,
        F.col("revenue") - F.col("budget"),
    ).alias("profit"),
    F.when(
        F.col("budget") > 0,
        (F.col("revenue") - F.col("budget")) / F.col("budget"),
    ).alias("roi"),
)

## Step 5. Produtos Gold

Constrói os produtos Gold a partir das entidades Silver e das métricas compartilhadas reconciliadas em `_eda/gold_contract_reconciliation`

### 5.1 movie_performance

Reúne os principais atributos e métricas de cada filme em uma visão individual de desempenho, mantendo uma linha por filme.

In [0]:
movie_performance_df = movie_metrics_df.select(
    "movie_id",
    "title",
    "original_title",
    "release_date",
    "release_year",
    "original_language",
    "status",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

### 5.2 movie_genre_performance

Associa os atributos e métricas de cada filme aos gêneros aos quais ele pertence, mantendo uma linha para cada relação filme–gênero.

Isso permite analisar o desempenho dos filmes sob a perspectiva de seus gêneros.

In [0]:
movie_genre_performance_df = movie_genre_df.join(
    movie_metrics_df,
    on="movie_id",
    how="inner",
).select(
    "movie_id",
    "genre_id",
    "genre_name",
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

### 5.3 movie_credit_participation
Reúne elenco e equipe em uma visão única das pessoas que participaram de cada filme `participation_type` diferencia participações de elenco (`cast`) e equipe técnica (`crew`), preservando os atributos específicos de cada tipo.

In [0]:
cast_participation_df = cast_credit_df.select(
    "movie_id",
    F.lit("cast").alias("participation_type"),
    "credit_id",
    "person_id",
    "person_name",
    "character",
    "cast_order",
    F.lit(None).cast("string").alias("department"),
    F.lit(None).cast("string").alias("job"),
)

crew_participation_df = crew_credit_df.select(
    "movie_id",
    F.lit("crew").alias("participation_type"),
    "credit_id",
    "person_id",
    "person_name",
    F.lit(None).cast("string").alias("character"),
    F.lit(None).cast("bigint").alias("cast_order"),
    "department",
    "job",
)

movie_credit_participation_df = (
    cast_participation_df.unionByName(crew_participation_df)
    .join(movie_df.select("movie_id", "title"), on="movie_id", how="inner")
    .select(
        "movie_id",
        "title",
        "participation_type",
        "credit_id",
        "person_id",
        "person_name",
        "character",
        "cast_order",
        "department",
        "job",
    )
)

### 5.4 movie_company_performance

Associa os atributos e métricas do filme a cada relação filme–produtora, mantendo uma linha por (movie_id, company_id)

In [0]:
movie_company_performance_df = movie_production_company_df.join(
    movie_metrics_df, on="movie_id", how="inner"
).select(
    "movie_id",
    "company_id",
    "company_name",
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

### 5.5 movie_country_performance

Associa os atributos e métricas do filme a cada relação filme–país, mantendo uma linha por `(movie_id, country_code)`

In [0]:
movie_country_performance_df = movie_production_country_df.join(
    movie_metrics_df, on="movie_id", how="inner"
).select(
    "movie_id",
    "country_code",
    "country_name",
    "original_language",
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

### 5.6 movie_language_profile

Cria uma visão unificada dos idiomas associados a cada filme. O produto diferencia o idioma original da obra dos idiomas falados no
filme por meio de `language_role`:
- `original`: idioma original registrado para o filme;
- `spoken`: idioma falado associado ao filme.

Assim, um mesmo filme pode possuir várias linhas sem perder o significado de cada relação com idioma.

In [0]:
original_language_df = movie_metrics_df.select(
    "movie_id",
    F.lit("original").alias("language_role"),
    F.col("original_language").alias("language_code"),
    F.lit(None).cast("string").alias("language_name"),
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

spoken_language_df = movie_spoken_language_df.join(
    movie_metrics_df,
    on="movie_id",
    how="inner",
).select(
    "movie_id",
    F.lit("spoken").alias("language_role"),
    "language_code",
    "language_name",
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

movie_language_profile_df = original_language_df.unionByName(spoken_language_df)

### 5.7 movie_keyword_performance

Associa os atributos e métricas de cada filme às keywords que descrevem seu conteúdo, mantendo uma linha para cada relação filme–keyword.
Isso permite analisar desempenho e características dos filmes a partir dos temas e conceitos representados por suas keywords.

In [0]:
movie_keyword_performance_df = movie_keyword_df.join(
    movie_metrics_df,
    on="movie_id",
    how="inner",
).select(
    "movie_id",
    "keyword_id",
    "keyword_name",
    "title",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "commercial_metrics_eligible",
    "profit",
    "roi",
    "popularity",
    "vote_average",
    "vote_count",
)

## Step 6. Materialização Gold

Materializa os produtos analíticos construídos nas etapas anteriores como tabelas do namespace Gold.

Cada DataFrame é associado explicitamente ao produto definido em `GOLD_CONTRACTS`, mantendo os nomes dos produtos como parte do contrato compartilhado da aplicação.

In [0]:
gold_dataframes = {
    "movie_performance": movie_performance_df,
    "movie_genre_performance": movie_genre_performance_df,
    "movie_credit_participation": movie_credit_participation_df,
    "movie_company_performance": movie_company_performance_df,
    "movie_country_performance": movie_country_performance_df,
    "movie_language_profile": movie_language_profile_df,
    "movie_keyword_performance": movie_keyword_performance_df,
}


gold_contract_names = {contract.name for contract in GOLD_CONTRACTS}

if set(gold_dataframes) != gold_contract_names:
    raise RuntimeError(
        "Gold Dataframes não correspondem aos contratos (gold_contract_names) "
        f"DataFrames: {sorted(gold_dataframes)}; "
        f"Contracts: {sorted(gold_contract_names)}"
    )

print(
    f"Iniciando o salvamento na camada gold: "
    f"{len(GOLD_CONTRACTS)} produtos -> {config.gold_namespace}"
)

for position, contract in enumerate(GOLD_CONTRACTS, start=1):
    table_name = f"{config.gold_namespace}.{contract.name}"

    print(f"[{position}/{len(GOLD_CONTRACTS)}] Materializando {contract.name}...")

    try:
        (
            gold_dataframes[contract.name]
            .write.format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        print(f"[{position}/{len(GOLD_CONTRACTS)}] Completo: {table_name}")

    except Exception as exc:
        print(f"[{position}/{len(GOLD_CONTRACTS)}] Falha: {table_name}")
        raise

## Step 7. Validação da materialização

Confirma que cada produto construído pelo notebook foi persistido na camada Gold com a mesma quantidade de registros produzida em memória.
Essa verificação garante que a materialização não perdeu nem acrescentou registros durante a escrita das tabelas.

In [0]:
print(f"Validando a meterialização na camada Gold: {len(GOLD_CONTRACTS)} produtos")

for position, contract in enumerate(GOLD_CONTRACTS, start=1):
    table_name = f"{config.gold_namespace}.{contract.name}"

    produced_count = gold_dataframes[contract.name].count()
    persisted_count = spark.table(table_name).count()

    print(
        f"[{position}/{len(GOLD_CONTRACTS)}] "
        f"{contract.name}: "
        f"produzidos = {produced_count}, "
        f"persistidos = {persisted_count}"
    )

    if produced_count != persisted_count:
        raise RuntimeError(
            f"Falha na validação de {table_name}: "
            f"produzidos = {produced_count}, "
            f"persistidos = {persisted_count}"
        )
    
print("Validação de materialização Gold concluída com sucesso")

## Step 8. Finalização

Registra a conclusão da construção dos produtos Gold após a materialização e a validação de todos os produtos definidos para a camada

In [0]:
print("Gold transformation completed successfully")
print(f"Silver Namespace: {config.silver_namespace}")
print(f"Gold Namespace: {config.gold_namespace}")
print(f"Products Materialized: {len(GOLD_CONTRACTS)}")